# 📈 FinSentLSTM — Multimodal Stock Direction Predictor

### Complete Google Colab Notebook — Run Everything Here!

> **Dual-branch LSTM that fuses 30-day OHLCV price sequences + news headline sentiment to predict next-day stock direction.**

---

## 🗺️ What This Notebook Does

| Step | Description |
|------|-------------|
| 1 | ✅ Enable GPU + Install dependencies |
| 2 | ✅ Write all source files to Colab disk |
| 3 | ✅ Preview & understand the data |
| 4 | ✅ Train the model (GPU accelerated) |
| 5 | ✅ Plot training curves |
| 6 | ✅ Run live inference on any ticker |
| 7 | ✅ Launch Streamlit dashboard with public URL |

---

⚡ **Before starting:** Go to `Runtime → Change Runtime Type → T4 GPU`

---

## ✅ Step 1 — GPU Check & Install Dependencies

In [ ]:
# ── Check GPU ──────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    print(f'🚀 GPU Ready: {torch.cuda.get_device_name(0)}')
    print(f'   Memory   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'   PyTorch  : {torch.__version__}')
else:
    print('⚠️  No GPU detected. Go to Runtime → Change Runtime Type → T4 GPU')
    print('   Training will still work on CPU but will be slower (~3-4x).')

In [ ]:
# ── Install dependencies ────────────────────────────────────────────
!pip install yfinance scikit-learn streamlit pyngrok plotly -q
print('✅ All dependencies installed!')

## ✅ Step 2 — Write Project Files to Colab

In [ ]:
import os
os.makedirs('models', exist_ok=True)
print('📁 Created models/ directory')

In [ ]:
%%writefile model.py
"""
FinSentLSTM — Multimodal Stock Predictor
=========================================
Architecture:
  Branch 1 → Price LSTM     : sequences of OHLCV data
  Branch 2 → Sentiment LSTM : sequences of tokenized news headlines
  Fusion    → Concatenate both hidden states → FC layers → Up/Down prediction
"""

import torch
import torch.nn as nn


class PriceLSTM(nn.Module):
    def __init__(self, input_size=5, hidden_size=128, num_layers=2, dropout=0.3):
        super(PriceLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out, (hn, _) = self.lstm(x)
        last_hidden   = hn[-1]
        return self.dropout(last_hidden)


class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size=10000, embed_dim=64, hidden_size=64,
                 num_layers=1, dropout=0.3, pad_idx=0):
        super(SentimentLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        embedded    = self.embedding(x)
        out, (hn, _) = self.lstm(embedded)
        last_hidden  = hn[-1]
        return self.dropout(last_hidden)


class FinSentLSTM(nn.Module):
    def __init__(
        self,
        price_input_size   = 5,
        price_hidden       = 128,
        price_layers       = 2,
        vocab_size         = 10000,
        embed_dim          = 64,
        sent_hidden        = 64,
        sent_layers        = 1,
        dropout            = 0.3,
        num_classes        = 2,
    ):
        super(FinSentLSTM, self).__init__()

        self.price_branch = PriceLSTM(
            input_size=price_input_size,
            hidden_size=price_hidden,
            num_layers=price_layers,
            dropout=dropout,
        )
        self.sentiment_branch = SentimentLSTM(
            vocab_size=vocab_size,
            embed_dim=embed_dim,
            hidden_size=sent_hidden,
            num_layers=sent_layers,
            dropout=dropout,
        )

        fused_size = price_hidden + sent_hidden

        self.classifier = nn.Sequential(
            nn.Linear(fused_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes),
        )

    def forward(self, price_seq, sent_seq):
        price_feat = self.price_branch(price_seq)
        sent_feat  = self.sentiment_branch(sent_seq)
        fused  = torch.cat([price_feat, sent_feat], dim=1)
        logits = self.classifier(fused)
        return logits


if __name__ == '__main__':
    model = FinSentLSTM()
    dummy_price = torch.randn(8, 30, 5)
    dummy_sent  = torch.randint(0, 10000, (8, 20))
    out = model(dummy_price, dummy_sent)
    print(f'✅ Output shape: {out.shape}')
    print(f'✅ Model params : {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
%%writefile data_pipeline.py
"""
data_pipeline.py — Downloads price data, builds vocab, creates DataLoaders
"""

import os
import re
import json
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.preprocessing import MinMaxScaler

import torch
from torch.utils.data import Dataset, DataLoader


def download_price_data(ticker: str, start: str, end: str) -> pd.DataFrame:
    try:
        import yfinance as yf
    except ImportError:
        raise ImportError('Run: pip install yfinance')
    df = yf.download(ticker, start=start, end=end, progress=False)
    if hasattr(df.columns, 'get_level_values'):
        df.columns = df.columns.get_level_values(0)
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()
    df.index = pd.to_datetime(df.index)
    print(f'✅ Downloaded {len(df)} rows for {ticker}')
    return df


def normalize_prices(df: pd.DataFrame):
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled = scaler.fit_transform(df.values)
    return scaled, scaler


def create_price_sequences(data: np.ndarray, seq_len: int = 30):
    X, y = [], []
    closes = data[:, 3]
    for i in range(len(data) - seq_len):
        X.append(data[i : i + seq_len])
        label = 1 if closes[i + seq_len] > closes[i + seq_len - 1] else 0
        y.append(label)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)


SAMPLE_HEADLINES = [
    'Apple reports record quarterly earnings beating analyst expectations',
    'Stock market rallies as Fed signals pause in rate hikes',
    'Tesla surges after strong delivery numbers exceed forecasts',
    'Tech stocks rise on strong GDP growth data',
    'Investors cheer inflation data showing price pressures easing',
    'Company announces major buyback program boosting investor confidence',
    'Strong jobs report signals robust economic growth ahead',
    'Pharmaceutical giant receives FDA approval for breakthrough drug',
    'Markets plunge amid recession fears and banking sector turmoil',
    'Fed raises rates again sparking selloff in growth stocks',
    'Inflation remains stubbornly high dampening investor sentiment',
    'Tech layoffs accelerate as companies cut costs aggressively',
    'Oil prices spike after OPEC production cuts announced',
    'Banking crisis deepens as regional lenders face deposit outflows',
    'Supply chain disruptions weigh on corporate profit margins',
    'Consumer confidence drops to multi-year low on economic worries',
    'Markets mixed as investors weigh economic data carefully',
    'Earnings season kicks off with mixed results across sectors',
    'Dollar steady against major currencies ahead of Fed meeting',
    'Analysts divided on outlook for technology sector this quarter',
]


def simple_tokenize(text: str) -> list:
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.split()


def build_vocab(headlines: list, max_vocab: int = 10000, min_freq: int = 1) -> dict:
    counter = Counter()
    for h in headlines:
        counter.update(simple_tokenize(h))
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, freq in counter.most_common(max_vocab - 2):
        if freq >= min_freq:
            vocab[word] = len(vocab)
    print(f'✅ Vocabulary size: {len(vocab)}')
    return vocab


def encode_headline(headline: str, vocab: dict, max_len: int = 20) -> list:
    tokens = simple_tokenize(headline)
    ids    = [vocab.get(t, vocab['<UNK>']) for t in tokens]
    ids    = ids[:max_len]
    ids   += [vocab['<PAD>']] * (max_len - len(ids))
    return ids


def save_vocab(vocab: dict, path: str):
    os.makedirs(os.path.dirname(path) if os.path.dirname(path) else '.', exist_ok=True)
    with open(path, 'w') as f:
        json.dump(vocab, f)
    print(f'✅ Vocab saved → {path}')


def load_vocab(path: str) -> dict:
    with open(path) as f:
        return json.load(f)


class StockDataset(Dataset):
    def __init__(self, price_X, price_y, headlines, vocab, sent_max_len=20):
        self.price_X      = torch.tensor(price_X, dtype=torch.float32)
        self.price_y      = torch.tensor(price_y, dtype=torch.long)
        self.vocab        = vocab
        self.sent_max_len = sent_max_len
        self.encoded_headlines = [
            encode_headline(h, vocab, sent_max_len) for h in headlines
        ]

    def __len__(self):
        return len(self.price_X)

    def __getitem__(self, idx):
        price_seq = self.price_X[idx]
        label     = self.price_y[idx]
        h_idx     = idx % len(self.encoded_headlines)
        sent_seq  = torch.tensor(self.encoded_headlines[h_idx], dtype=torch.long)
        return price_seq, sent_seq, label


def get_dataloaders(
    ticker='AAPL', start='2018-01-01', end='2024-01-01',
    seq_len=30, batch_size=32, val_split=0.15, test_split=0.10,
    vocab_path='models/vocab.json',
):
    df             = download_price_data(ticker, start, end)
    scaled, scaler = normalize_prices(df)
    X, y           = create_price_sequences(scaled, seq_len)

    N        = len(X)
    val_cut  = int(N * (1 - val_split - test_split))
    test_cut = int(N * (1 - test_split))

    X_train, y_train = X[:val_cut],         y[:val_cut]
    X_val,   y_val   = X[val_cut:test_cut], y[val_cut:test_cut]
    X_test,  y_test  = X[test_cut:],        y[test_cut:]

    print(f'📊 Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

    vocab = build_vocab(SAMPLE_HEADLINES)
    save_vocab(vocab, vocab_path)

    train_ds = StockDataset(X_train, y_train, SAMPLE_HEADLINES, vocab)
    val_ds   = StockDataset(X_val,   y_val,   SAMPLE_HEADLINES, vocab)
    test_ds  = StockDataset(X_test,  y_test,  SAMPLE_HEADLINES, vocab)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=0)

    return train_loader, val_loader, test_loader, scaler, vocab

In [ ]:
%%writefile train.py
"""
train.py — Complete training pipeline for FinSentLSTM
"""

import os
import json
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from model import FinSentLSTM
from data_pipeline import get_dataloaders

CONFIG = {
    'ticker'      : 'AAPL',
    'start_date'  : '2018-01-01',
    'end_date'    : '2024-01-01',
    'seq_len'     : 30,
    'batch_size'  : 32,
    'price_hidden': 128,
    'price_layers': 2,
    'vocab_size'  : 10000,
    'embed_dim'   : 64,
    'sent_hidden' : 64,
    'sent_layers' : 1,
    'dropout'     : 0.3,
    'epochs'      : 50,
    'lr'          : 1e-3,
    'weight_decay': 1e-5,
    'patience'    : 8,
    'lr_patience' : 4,
    'lr_factor'   : 0.5,
    'min_lr'      : 1e-6,
    'model_path'  : 'models/best_model.pt',
    'vocab_path'  : 'models/vocab.json',
    'history_path': 'models/history.json',
}


def get_device():
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print(f'🚀 Using GPU: {torch.cuda.get_device_name(0)}')
    else:
        device = torch.device('cpu')
        print('💻 Using CPU')
    return device


def compute_metrics(y_true, y_pred):
    return {
        'accuracy' : accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall'   : recall_score(y_true, y_pred, zero_division=0),
        'f1'       : f1_score(y_true, y_pred, zero_division=0),
    }


def run_epoch(model, loader, criterion, optimizer, device, training=True):
    model.train() if training else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for price_seq, sent_seq, labels in loader:
            price_seq = price_seq.to(device)
            sent_seq  = sent_seq.to(device)
            labels    = labels.to(device)
            if training:
                optimizer.zero_grad()
            logits = model(price_seq, sent_seq)
            loss   = criterion(logits, labels)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_loss += loss.item()
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader)
    metrics  = compute_metrics(all_labels, all_preds)
    return avg_loss, metrics


def train():
    os.makedirs('models', exist_ok=True)
    device = get_device()

    print('\n📥 Loading data...')
    train_loader, val_loader, test_loader, scaler, vocab = get_dataloaders(
        ticker=CONFIG['ticker'], start=CONFIG['start_date'], end=CONFIG['end_date'],
        seq_len=CONFIG['seq_len'], batch_size=CONFIG['batch_size'],
        vocab_path=CONFIG['vocab_path'],
    )

    model = FinSentLSTM(
        price_hidden=CONFIG['price_hidden'], price_layers=CONFIG['price_layers'],
        vocab_size=len(vocab), embed_dim=CONFIG['embed_dim'],
        sent_hidden=CONFIG['sent_hidden'], sent_layers=CONFIG['sent_layers'],
        dropout=CONFIG['dropout'],
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'\n🧠 Model Parameters: {total_params:,}')

    all_labels = torch.cat([labels for _, _, labels in train_loader]).numpy()
    n_down = (all_labels == 0).sum()
    n_up   = (all_labels == 1).sum()
    weight = torch.tensor([n_up / n_down, 1.0], dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weight)

    optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=CONFIG['lr_patience'],
                                  factor=CONFIG['lr_factor'], min_lr=CONFIG['min_lr'])

    history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[], 'train_f1':[], 'val_f1':[], 'lr':[]}
    best_val_acc, patience_counter = 0.0, 0

    print('\n🏋️  Starting Training...\n')
    print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Val Loss":>8} | {"Train Acc":>9} | {"Val Acc":>7} | {"Val F1":>6} | {"LR":>8}')
    print('-' * 75)

    for epoch in range(1, CONFIG['epochs'] + 1):
        t0 = time.time()
        train_loss, train_m = run_epoch(model, train_loader, criterion, optimizer, device, training=True)
        val_loss,   val_m   = run_epoch(model, val_loader,   criterion, optimizer, device, training=False)
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(train_loss)
        history['val_loss'  ].append(val_loss)
        history['train_acc' ].append(train_m['accuracy'])
        history['val_acc'   ].append(val_m['accuracy'])
        history['train_f1'  ].append(train_m['f1'])
        history['val_f1'    ].append(val_m['f1'])
        history['lr'        ].append(current_lr)

        elapsed = time.time() - t0
        print(f'{epoch:>6} | {train_loss:>10.4f} | {val_loss:>8.4f} | '
              f'{train_m["accuracy"]:>9.4f} | {val_m["accuracy"]:>7.4f} | '
              f'{val_m["f1"]:>6.4f} | {current_lr:>8.2e}  [{elapsed:.1f}s]')

        if val_m['accuracy'] > best_val_acc:
            best_val_acc = val_m['accuracy']
            torch.save({
                'epoch': epoch, 'model_state': model.state_dict(),
                'optimizer': optimizer.state_dict(), 'val_acc': best_val_acc,
                'config': CONFIG,
            }, CONFIG['model_path'])
            print(f'         💾 Saved best model (val_acc={best_val_acc:.4f})')
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= CONFIG['patience']:
            print(f'\n⛔ Early stopping at epoch {epoch}')
            break

    with open(CONFIG['history_path'], 'w') as f:
        json.dump(history, f)
    print(f'\n📈 History saved → {CONFIG["history_path"]}')

    print('\n🧪 Loading best model for test evaluation...')
    checkpoint = torch.load(CONFIG['model_path'], map_location=device)
    model.load_state_dict(checkpoint['model_state'])
    test_loss, test_m = run_epoch(model, test_loader, criterion, optimizer, device, training=False)

    print(f'\n{"="*40}')
    print(f'  📊 TEST RESULTS')
    print(f'{"="*40}')
    print(f'  Loss      : {test_loss:.4f}')
    print(f'  Accuracy  : {test_m["accuracy"]:.4f}')
    print(f'  Precision : {test_m["precision"]:.4f}')
    print(f'  Recall    : {test_m["recall"]:.4f}')
    print(f'  F1 Score  : {test_m["f1"]:.4f}')
    print(f'{"="*40}\n')

    return model, history


if __name__ == '__main__':
    train()

In [ ]:
%%writefile inference.py
"""
inference.py — Real-time prediction pipeline
"""

import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from data_pipeline import download_price_data, normalize_prices, encode_headline, load_vocab
from model import FinSentLSTM


def load_model(model_path='models/best_model.pt', vocab_path='models/vocab.json', device='cpu'):
    device     = torch.device(device)
    vocab      = load_vocab(vocab_path)
    checkpoint = torch.load(model_path, map_location=device)
    config     = checkpoint['config']

    model = FinSentLSTM(
        price_hidden=config['price_hidden'], price_layers=config['price_layers'],
        vocab_size=len(vocab), embed_dim=config['embed_dim'],
        sent_hidden=config['sent_hidden'], sent_layers=config['sent_layers'],
        dropout=0.0,
    ).to(device)

    model.load_state_dict(checkpoint['model_state'])
    model.eval()
    print(f'✅ Model loaded (val_acc={checkpoint["val_acc"]:.4f})')
    return model, vocab, config


def predict_live(ticker, headline, model, vocab, seq_len=30, device='cpu'):
    device = torch.device(device)
    from datetime import datetime, timedelta
    end   = datetime.today().strftime('%Y-%m-%d')
    start = (datetime.today() - timedelta(days=seq_len * 2)).strftime('%Y-%m-%d')

    df = download_price_data(ticker, start, end)
    if len(df) < seq_len:
        raise ValueError(f'Not enough data: got {len(df)} rows, need {seq_len}.')

    recent_df  = df.tail(seq_len)
    last_close = float(recent_df['Close'].iloc[-1])
    scaled, _  = normalize_prices(recent_df)
    price_tensor = torch.tensor(scaled, dtype=torch.float32).unsqueeze(0).to(device)

    sent_ids    = encode_headline(headline, vocab, max_len=20)
    sent_tensor = torch.tensor(sent_ids, dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(price_tensor, sent_tensor)
        probs  = F.softmax(logits, dim=1).squeeze().cpu().numpy()

    down_prob, up_prob = float(probs[0]), float(probs[1])
    prediction  = 'UP 📈' if up_prob > down_prob else 'DOWN 📉'
    confidence  = max(up_prob, down_prob)
    price_change = float((recent_df['Close'].iloc[-1] - recent_df['Close'].iloc[0])
                         / recent_df['Close'].iloc[0] * 100)

    return {
        'prediction': prediction, 'confidence': confidence,
        'up_prob': up_prob, 'down_prob': down_prob,
        'last_close': last_close, 'price_change': price_change,
        'ticker': ticker, 'headline': headline,
    }


def get_price_history(ticker: str, days: int = 90) -> pd.DataFrame:
    from datetime import datetime, timedelta
    end   = datetime.today().strftime('%Y-%m-%d')
    start = (datetime.today() - timedelta(days=days)).strftime('%Y-%m-%d')
    df    = download_price_data(ticker, start, end)
    df    = df.reset_index()
    df['Date'] = pd.to_datetime(df['Date'])
    return df


def get_training_history(history_path='models/history.json') -> dict:
    with open(history_path) as f:
        return json.load(f)

In [ ]:
%%writefile app.py
"""
app.py  —  FinSentLSTM Streamlit Dashboard
"""

import os, sys, json
import numpy as np
import pandas as pd
import streamlit as st
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import datetime, timedelta

st.set_page_config(page_title='FinSentLSTM', page_icon='📈', layout='wide')

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;700&family=Syne:wght@400;700;800&display=swap');
:root{--bg-primary:#070b14;--bg-card:#0d1424;--bg-card2:#111827;--accent-green:#00ff87;--accent-red:#ff4d6d;--accent-blue:#3b82f6;--accent-gold:#f59e0b;--text-primary:#e2e8f0;--text-muted:#64748b;--border:#1e293b;}
html,body,.stApp{background-color:var(--bg-primary)!important;font-family:'Syne',sans-serif;color:var(--text-primary);}
#MainMenu,footer,header{visibility:hidden;}.block-container{padding:1.5rem 2rem;max-width:1400px;}
[data-testid="stSidebar"]{background:var(--bg-card)!important;border-right:1px solid var(--border);}
[data-testid="stSidebar"]*{color:var(--text-primary)!important;}
[data-testid="metric-container"]{background:var(--bg-card2);border:1px solid var(--border);border-radius:12px;padding:1rem;}
[data-testid="stMetricValue"]{font-family:'JetBrains Mono',monospace;font-size:1.6rem!important;font-weight:700;}
.stTextInput>div>div>input,.stSelectbox>div>div{background:var(--bg-card2)!important;border:1px solid var(--border)!important;border-radius:8px!important;color:var(--text-primary)!important;}
.stButton>button{background:linear-gradient(135deg,#3b82f6,#1d4ed8)!important;color:white!important;border:none!important;border-radius:8px!important;font-weight:700!important;padding:0.6rem 2rem!important;}
.pred-card-up{background:linear-gradient(135deg,#052e16,#0d2a1a);border:1px solid #00ff87;border-radius:16px;padding:2rem;text-align:center;box-shadow:0 0 40px rgba(0,255,135,.1);}
.pred-card-down{background:linear-gradient(135deg,#2d0a14,#3d0f1a);border:1px solid #ff4d6d;border-radius:16px;padding:2rem;text-align:center;box-shadow:0 0 40px rgba(255,77,109,.1);}
.pred-label{font-size:3rem;font-weight:800;font-family:'JetBrains Mono',monospace;}
.info-box{background:var(--bg-card2);border:1px solid var(--border);border-left:3px solid var(--accent-blue);border-radius:8px;padding:1rem 1.2rem;margin:.5rem 0;font-size:.88rem;color:#94a3b8;font-family:'JetBrains Mono',monospace;}
.section-title{font-size:1.1rem;font-weight:700;color:#94a3b8;letter-spacing:.1em;text-transform:uppercase;margin:1.5rem 0 1rem;font-family:'JetBrains Mono',monospace;border-bottom:1px solid var(--border);padding-bottom:.5rem;}
</style>""", unsafe_allow_html=True)

PLOTLY_LAYOUT = dict(paper_bgcolor='#070b14',plot_bgcolor='#0d1424',
    font=dict(family='JetBrains Mono',color='#e2e8f0'),
    xaxis=dict(gridcolor='#1e293b',showgrid=True,color='#64748b'),
    yaxis=dict(gridcolor='#1e293b',showgrid=True,color='#64748b'),
    margin=dict(l=10,r=10,t=40,b=10))

POPULAR_TICKERS = ['AAPL','TSLA','GOOGL','MSFT','AMZN','NVDA','META','NFLX','AMD','JPM']
SAMPLE_HEADLINES = [
    'Fed signals rate pause boosting investor confidence in equities',
    'Strong earnings season drives tech stocks to record highs',
    'Inflation data shows continued easing pressuring bond yields',
    'Banking sector faces renewed scrutiny after credit concerns emerge',
    'Oil prices spike on OPEC supply cut announcement',
    'Consumer spending remains resilient defying recession fears',
    'Tech layoffs continue as companies prioritize profitability',
    'Markets fall on weak GDP data raising slowdown concerns',
]

@st.cache_resource(show_spinner=False)
def load_model_cached():
    try:
        from inference import load_model
        model, vocab, config = load_model('models/best_model.pt','models/vocab.json')
        return model, vocab, config, None
    except FileNotFoundError:
        return None, None, None, 'Model not trained yet.'
    except Exception as e:
        return None, None, None, str(e)

def make_candlestick_chart(df, ticker):
    fig = go.Figure()
    colors = ['#00ff87' if c>=o else '#ff4d6d' for c,o in zip(df['Close'],df['Open'])]
    fig.add_trace(go.Candlestick(x=df['Date'],open=df['Open'],high=df['High'],
        low=df['Low'],close=df['Close'],name=ticker,
        increasing_line_color='#00ff87',decreasing_line_color='#ff4d6d',
        increasing_fillcolor='rgba(0,255,135,0.15)',decreasing_fillcolor='rgba(255,77,109,0.15)'))
    fig.add_trace(go.Bar(x=df['Date'],y=df['Volume'],name='Volume',
        marker_color=colors,opacity=0.3,yaxis='y2'))
    fig.update_layout(**PLOTLY_LAYOUT,title=f'{ticker} — Price & Volume',
        yaxis2=dict(overlaying='y',side='right',showgrid=False,color='#64748b'),
        xaxis_rangeslider_visible=False,height=420)
    return fig

def make_training_charts(history):
    epochs = list(range(1,len(history['train_loss'])+1))
    fig = make_subplots(rows=1,cols=2,subplot_titles=['Loss Curve','Accuracy Curve'])
    fig.add_trace(go.Scatter(x=epochs,y=history['train_loss'],name='Train Loss',line=dict(color='#3b82f6',width=2)),row=1,col=1)
    fig.add_trace(go.Scatter(x=epochs,y=history['val_loss'],name='Val Loss',line=dict(color='#f59e0b',width=2,dash='dot')),row=1,col=1)
    fig.add_trace(go.Scatter(x=epochs,y=history['train_acc'],name='Train Acc',line=dict(color='#00ff87',width=2)),row=1,col=2)
    fig.add_trace(go.Scatter(x=epochs,y=history['val_acc'],name='Val Acc',line=dict(color='#ff4d6d',width=2,dash='dot')),row=1,col=2)
    fig.update_layout(**PLOTLY_LAYOUT,height=360,showlegend=True)
    return fig

def make_gauge(up_prob, down_prob):
    fig = go.Figure(go.Indicator(mode='gauge+number',value=up_prob*100,
        title=dict(text='UP Probability (%)',font=dict(color='#94a3b8',size=14)),
        number=dict(font=dict(color='#00ff87',size=40,family='JetBrains Mono'),suffix='%'),
        gauge=dict(axis=dict(range=[0,100]),bar=dict(color='#00ff87'),
            steps=[dict(range=[0,40],color='#1a0a10'),dict(range=[40,60],color='#111827'),dict(range=[60,100],color='#0a1f10')],
            threshold=dict(line=dict(color='#3b82f6',width=3),value=50),
            bgcolor='#0d1424',bordercolor='#1e293b')))
    fig.update_layout(paper_bgcolor='#070b14',height=260,margin=dict(l=20,r=20,t=40,b=10))
    return fig

# ── Sidebar
with st.sidebar:
    st.markdown("<div style='text-align:center;padding:1rem 0'><div style='font-size:2.5rem'>📈</div><div style='font-weight:800;font-size:1.2rem;color:#e2e8f0'>FinSentLSTM</div><div style='font-size:.75rem;color:#64748b'>Multimodal Stock Predictor</div></div><hr style='border-color:#1e293b;margin:1rem 0'>", unsafe_allow_html=True)
    st.markdown('**🎯 Select Stock**')
    ticker = st.selectbox('Ticker', POPULAR_TICKERS, index=0, label_visibility='collapsed')
    custom_ticker = st.text_input('Or enter custom ticker:', placeholder='e.g. RELIANCE.NS')
    if custom_ticker.strip(): ticker = custom_ticker.strip().upper()
    st.markdown('**📰 Today\'s Headline**')
    headline = st.selectbox('Select a headline:', SAMPLE_HEADLINES, label_visibility='collapsed')
    custom_hl = st.text_area('Custom headline:', placeholder='Enter your own news headline...', height=80)
    if custom_hl.strip(): headline = custom_hl.strip()
    st.markdown('**⚙️ Settings**')
    seq_len    = st.slider('Look-back window (days):', 10, 60, 30)
    chart_days = st.slider('Chart history (days):', 30, 365, 90)
    st.markdown("<hr style='border-color:#1e293b'>", unsafe_allow_html=True)
    predict_btn = st.button('🚀 Run Prediction', use_container_width=True)
    st.markdown("<div class='info-box'>⚠️ Educational project only.<br>Not financial advice.</div>", unsafe_allow_html=True)

# ── Header
st.markdown(f"<div style='background:linear-gradient(135deg,#0d1424,#0f1e3d);border:1px solid #1e3a5f;border-radius:16px;padding:2rem 2.5rem;margin-bottom:2rem'><div style='font-size:2.2rem;font-weight:800;background:linear-gradient(135deg,#e2e8f0,#3b82f6);-webkit-background-clip:text;-webkit-text-fill-color:transparent'>FinSentLSTM Dashboard</div><div style='color:#64748b;font-size:.95rem;font-family:JetBrains Mono,monospace'>Multimodal LSTM · Price + News Sentiment Fusion · PyTorch &nbsp;|&nbsp; Ticker: <span style='color:#3b82f6'>{ticker}</span> &nbsp;|&nbsp; {datetime.now().strftime('%d %b %Y, %H:%M')}</div></div>", unsafe_allow_html=True)

model, vocab, config, error = load_model_cached()
tab1, tab2, tab3, tab4 = st.tabs(['📊 Live Prediction','📈 Price Chart','🏋️ Training Metrics','🧠 Model Info'])

with tab1:
    if error:
        st.error(f'⚠️ {error}')
    else:
        if predict_btn:
            with st.spinner(f'🔍 Fetching {ticker} data and running inference...'):
                try:
                    from inference import predict_live
                    result = predict_live(ticker=ticker, headline=headline, model=model, vocab=vocab, seq_len=seq_len)
                    st.session_state['result'] = result
                except Exception as e:
                    st.error(f'Prediction failed: {e}')
                    st.session_state['result'] = None
        if 'result' in st.session_state and st.session_state['result']:
            result = st.session_state['result']
            is_up = 'UP' in result['prediction']
            card_class = 'pred-card-up' if is_up else 'pred-card-down'
            color = '#00ff87' if is_up else '#ff4d6d'
            arrow = '▲' if is_up else '▼'
            st.markdown(f"<div class='{card_class}'><div class='pred-label' style='color:{color}'>{arrow} {result['prediction']}</div><div style='color:#94a3b8;font-family:JetBrains Mono,monospace;margin-top:.5rem'>Confidence: {result['confidence']*100:.1f}% &nbsp;|&nbsp; Last Close: ${result['last_close']:.2f} &nbsp;|&nbsp; 30d Δ: {result['price_change']:+.2f}%</div></div><br>", unsafe_allow_html=True)
            c1,c2,c3,c4 = st.columns(4)
            c1.metric('📈 UP Prob',  f"{result['up_prob']*100:.1f}%")
            c2.metric('📉 DOWN Prob',f"{result['down_prob']*100:.1f}%")
            c3.metric('💰 Last Close',f"${result['last_close']:.2f}")
            c4.metric('📅 30d Δ',    f"{result['price_change']:+.2f}%")
            g1,g2 = st.columns([1,2])
            with g1: st.plotly_chart(make_gauge(result['up_prob'],result['down_prob']),use_container_width=True)
            with g2: st.markdown(f"<div class='info-box'><b style='color:#e2e8f0'>📰 Headline:</b><br>{result['headline']}</div><div class='info-box'><b style='color:#e2e8f0'>🧠 Model:</b><br>Price LSTM({config['price_hidden']}) × {config['price_layers']} layers + Sentiment LSTM({config['sent_hidden']}) → FC(192→128→64→2)</div>", unsafe_allow_html=True)
        else:
            st.markdown("<div style='text-align:center;padding:4rem;color:#374151'><div style='font-size:4rem'>📊</div><div style='font-size:1.2rem;margin-top:1rem'>Select a ticker and click <b>Run Prediction</b></div></div>", unsafe_allow_html=True)

with tab2:
    with st.spinner(f'📈 Loading {chart_days} days of {ticker} data...'):
        try:
            from inference import get_price_history
            df_chart = get_price_history(ticker, days=chart_days)
            st.plotly_chart(make_candlestick_chart(df_chart, ticker), use_container_width=True)
            latest,prev = df_chart.iloc[-1],df_chart.iloc[-2]
            ch = latest['Close']-prev['Close']; ch_pct = ch/prev['Close']*100
            s1,s2,s3,s4 = st.columns(4)
            s1.metric('Open',f"${latest['Open']:.2f}"); s2.metric('High',f"${latest['High']:.2f}")
            s3.metric('Low',f"${latest['Low']:.2f}"); s4.metric('Close',f"${latest['Close']:.2f}",delta=f'{ch_pct:+.2f}%')
        except Exception as e: st.error(f'Could not load chart: {e}')

with tab3:
    try:
        from inference import get_training_history
        history = get_training_history('models/history.json')
        st.plotly_chart(make_training_charts(history), use_container_width=True)
        m1,m2,m3,m4 = st.columns(4)
        m1.metric('Best Val Accuracy',f"{max(history['val_acc'])*100:.1f}%")
        m2.metric('Best Val F1',f"{max(history['val_f1']):.4f}")
        m3.metric('Best Epoch',int(np.argmax(history['val_acc']))+1)
        m4.metric('Total Epochs',len(history['train_loss']))
    except FileNotFoundError:
        st.info('💡 Train the model first to see training curves.')

with tab4:
    c1,c2 = st.columns(2)
    with c1:
        st.markdown("<div class='info-box'><b style='color:#3b82f6'>🏗️ FinSentLSTM Architecture</b><br><br><b>Branch 1 — Price LSTM</b><br>Input: OHLCV sequences (30 days × 5 features)<br>Layers: 2-layer stacked LSTM (hidden=128)<br>Output: 128-dim embedding<br><br><b>Branch 2 — Sentiment LSTM</b><br>Input: Tokenized headlines (20 tokens)<br>Layers: Embedding(64) → LSTM(64)<br>Output: 64-dim embedding<br><br><b>Fusion</b><br>Concat(192) → FC(128) → FC(64) → FC(2)</div>", unsafe_allow_html=True)
    with c2:
        st.markdown("<div class='info-box'><b style='color:#00ff87'>🛠️ Training Details</b><br><br>Optimizer: AdamW (lr=1e-3)<br>Loss: Weighted CrossEntropy<br>Scheduler: ReduceLROnPlateau<br>Grad Clip: max_norm=1.0<br>Early Stop: patience=8<br>Dropout: 0.3<br><br><b style='color:#f59e0b'>📊 Data Pipeline</b><br><br>Price: Yahoo Finance (yfinance)<br>Normalization: MinMaxScaler<br>Label: Next-day close direction<br>Seq length: 30 trading days</div>", unsafe_allow_html=True)

In [ ]:
# Verify all files are created
import os
files = ['model.py', 'data_pipeline.py', 'train.py', 'inference.py', 'app.py']
print('📁 Project files:')
for f in files:
    status = '✅' if os.path.exists(f) else '❌'
    print(f'   {status} {f}')
print('\n📁 Directories:')
print(f'   ✅ models/' if os.path.exists('models') else '   ❌ models/')

## ✅ Step 3 — Preview the Data

In [ ]:
# ── Preview stock data from Yahoo Finance ──────────────────────────
from data_pipeline import download_price_data, normalize_prices, create_price_sequences
import matplotlib.pyplot as plt

df = download_price_data('AAPL', '2023-01-01', '2024-01-01')
print('\n📊 Last 5 rows of AAPL data:')
print(df.tail())
print(f'\nShape: {df.shape}')
print(f'Date range: {df.index[0].date()} → {df.index[-1].date()}')

In [ ]:
# ── Plot price data ────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 7))
fig.patch.set_facecolor('#070b14')

for ax in axes:
    ax.set_facecolor('#0d1424')
    ax.tick_params(colors='#94a3b8')
    for spine in ax.spines.values():
        spine.set_color('#1e293b')

# Close price
axes[0].plot(df.index, df['Close'], color='#3b82f6', linewidth=2, label='Close')
axes[0].fill_between(df.index, df['Close'], alpha=0.1, color='#3b82f6')
axes[0].set_title('AAPL Close Price (2023)', color='#e2e8f0', fontsize=13, pad=10)
axes[0].set_ylabel('Price ($)', color='#94a3b8')
axes[0].legend(labelcolor='#94a3b8')
axes[0].grid(color='#1e293b', linewidth=0.5)

# Volume
colors = ['#00ff87' if c >= o else '#ff4d6d' for c, o in zip(df['Close'], df['Open'])]
axes[1].bar(df.index, df['Volume'], color=colors, alpha=0.7)
axes[1].set_title('Volume', color='#e2e8f0', fontsize=13, pad=10)
axes[1].set_ylabel('Volume', color='#94a3b8')
axes[1].grid(color='#1e293b', linewidth=0.5)

plt.tight_layout(pad=2)
plt.savefig('data_preview.png', dpi=120, bbox_inches='tight', facecolor='#070b14')
plt.show()
print('✅ Data preview plotted!')

In [ ]:
# ── Verify DataLoader output shapes ───────────────────────────────
from data_pipeline import get_dataloaders

train_loader, val_loader, test_loader, scaler, vocab = get_dataloaders(
    ticker='AAPL', start='2020-01-01', end='2024-01-01'
)

price_batch, sent_batch, label_batch = next(iter(train_loader))
print(f'\n📦 Batch shapes:')
print(f'   Price  : {price_batch.shape}   → (batch, seq_len, 5 features)')
print(f'   Sent   : {sent_batch.shape}    → (batch, 20 tokens)')
print(f'   Labels : {label_batch.shape}   → (batch,)  0=DOWN, 1=UP')
print(f'   Sample labels: {label_batch[:8].tolist()}')

## ✅ Step 4 — Train the Model 🏋️

In [ ]:
# ── Model sanity check before training ────────────────────────────
import torch
from model import FinSentLSTM

model = FinSentLSTM()
dummy_price = torch.randn(8, 30, 5)
dummy_sent  = torch.randint(0, 100, (8, 20))
out = model(dummy_price, dummy_sent)
print(f'✅ Model forward pass OK!')
print(f'   Output shape : {out.shape}  → (batch=8, classes=2)')
print(f'   Total params : {sum(p.numel() for p in model.parameters()):,}')
print(f'\n🏗️ Architecture:')
print(model)

In [ ]:
# ── TRAIN THE MODEL ────────────────────────────────────────────────
# This will take ~5-15 minutes depending on GPU/CPU
# Watch the Val Acc column — target is 55-63%

from train import train
model, history = train()

## ✅ Step 5 — Plot Training Curves 📈

In [ ]:
import json
import matplotlib.pyplot as plt

with open('models/history.json') as f:
    history = json.load(f)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#070b14')

titles  = ['Loss Curve', 'Accuracy Curve', 'F1 Score Curve']
train_k = ['train_loss', 'train_acc', 'train_f1']
val_k   = ['val_loss',   'val_acc',   'val_f1']
colors  = [('#3b82f6','#f59e0b'), ('#00ff87','#ff4d6d'), ('#a78bfa','#fb923c')]

epochs = range(1, len(history['train_loss']) + 1)

for ax, title, tk, vk, (tc, vc) in zip(axes, titles, train_k, val_k, colors):
    ax.set_facecolor('#0d1424')
    ax.tick_params(colors='#94a3b8')
    for spine in ax.spines.values(): spine.set_color('#1e293b')
    ax.plot(epochs, history[tk], color=tc, label='Train', linewidth=2)
    ax.plot(epochs, history[vk], color=vc, label='Val',   linewidth=2, linestyle='--')
    ax.set_title(title, color='#e2e8f0', fontsize=13)
    ax.legend(labelcolor='#94a3b8')
    ax.grid(color='#1e293b', linewidth=0.5)

plt.suptitle('FinSentLSTM Training History', color='#e2e8f0', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight', facecolor='#070b14')
plt.show()

print(f'\n📊 Training Summary:')
print(f'   Best Val Accuracy : {max(history["val_acc"])*100:.2f}%')
print(f'   Best Val F1       : {max(history["val_f1"]):.4f}')
print(f'   Total Epochs      : {len(history["train_loss"])}')
print(f'   Best Epoch        : {history["val_acc"].index(max(history["val_acc"]))+1}')

## ✅ Step 6 — Live Inference 🔮

In [ ]:
# ── Load trained model and run inference ──────────────────────────
from inference import load_model, predict_live

model, vocab, config = load_model()

# ── Try multiple tickers and headlines ────────────────────────────
test_cases = [
    ('AAPL', 'Apple beats earnings with record iPhone sales and strong guidance'),
    ('TSLA', 'Tesla misses delivery targets amid production challenges'),
    ('MSFT', 'Microsoft Azure cloud revenue surges 30% beating expectations'),
    ('NVDA', 'Nvidia reports record data center revenue driven by AI chip demand'),
]

print('\n' + '='*65)
print(f'{"TICKER":<8} {"PREDICTION":<12} {"CONF":>6} {"UP%":>6} {"DOWN%":>6} {"CLOSE":>8}')
print('='*65)

for ticker, headline in test_cases:
    try:
        result = predict_live(ticker=ticker, headline=headline, model=model, vocab=vocab)
        print(f"{result['ticker']:<8} {result['prediction']:<14} {result['confidence']*100:>5.1f}% "
              f"{result['up_prob']*100:>5.1f}% {result['down_prob']*100:>5.1f}% "
              f"${result['last_close']:>7.2f}")
    except Exception as e:
        print(f'{ticker:<8} Error: {e}')

print('='*65)

In [ ]:
# ── Detailed prediction for one ticker ────────────────────────────
from inference import predict_live, get_price_history

TICKER   = 'AAPL'    # ← Change this!
HEADLINE = 'Apple reports record quarterly revenue beating all analyst expectations'

result = predict_live(ticker=TICKER, headline=HEADLINE, model=model, vocab=vocab)

print('\n' + '='*50)
print(f'  📊 PREDICTION DETAILS: {TICKER}')
print('='*50)
for k, v in result.items():
    if isinstance(v, float):
        print(f'  {k:<15}: {v:.4f}')
    else:
        print(f'  {k:<15}: {v}')
print('='*50)

# Plot recent price history
df_hist = get_price_history(TICKER, days=60)

fig, ax = plt.subplots(figsize=(14, 4))
fig.patch.set_facecolor('#070b14')
ax.set_facecolor('#0d1424')
ax.tick_params(colors='#94a3b8')
for spine in ax.spines.values(): spine.set_color('#1e293b')

color = '#00ff87' if 'UP' in result['prediction'] else '#ff4d6d'
ax.plot(df_hist['Date'], df_hist['Close'], color=color, linewidth=2)
ax.fill_between(df_hist['Date'], df_hist['Close'], alpha=0.1, color=color)
ax.set_title(f'{TICKER} — Recent 60 Days | Prediction: {result["prediction"]} ({result["confidence"]*100:.1f}% confidence)',
             color='#e2e8f0', fontsize=12)
ax.set_ylabel('Price ($)', color='#94a3b8')
ax.grid(color='#1e293b', linewidth=0.5)
plt.tight_layout()
plt.show()

## ✅ Step 7 — Launch Streamlit Dashboard 🚀

> This creates a **public URL** you can open in any browser!

In [ ]:
# ── Install tunnel dependencies ───────────────────────────────────
!pip install streamlit pyngrok -q
print('✅ Streamlit + ngrok ready')

In [ ]:
# ── (Optional) Set your ngrok auth token for a stable URL ─────────
# Get FREE token at: https://dashboard.ngrok.com/signup (takes 30 seconds)
# Then paste it below:

NGROK_TOKEN = ''   # ← Paste your token here (optional but recommended)

if NGROK_TOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    print('✅ ngrok token set!')
else:
    print('ℹ️  No token set — will use anonymous tunnel (may show ngrok warning page)')

In [ ]:
# ── Launch the Streamlit app ───────────────────────────────────────
import subprocess, time, os
from pyngrok import ngrok

# Kill any existing processes on port 8501
os.system('fuser -k 8501/tcp 2>/dev/null || true')
time.sleep(1)

# Start Streamlit in background
proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port=8501',
     '--server.headless=true',
     '--server.enableCORS=false',
     '--server.enableXsrfProtection=false'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print('⏳ Starting Streamlit server...')
time.sleep(5)

# Create public tunnel
try:
    ngrok.kill()  # Kill any existing ngrok tunnels
except:
    pass

public_url = ngrok.connect(8501)
print('\n' + '='*60)
print(f'  🌍 STREAMLIT APP IS LIVE!')
print(f'  🔗 URL: {public_url}')
print('='*60)
print('\n  👆 Click the link above to open the dashboard')
print('  ⏹️  Run the next cell to stop the server')

In [ ]:
# ── Stop the server when done ──────────────────────────────────────
# Run this cell to shut everything down
try:
    proc.terminate()
    ngrok.kill()
    print('✅ Server stopped.')
except:
    print('ℹ️  Already stopped.')

---

## 💾 Save Your Model to Google Drive (Optional)

Save the trained model so you don't lose it when the Colab session ends!

In [ ]:
# ── Save to Google Drive ───────────────────────────────────────────
import shutil

from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/FinSentLSTM'
os.makedirs(SAVE_DIR, exist_ok=True)

shutil.copy('models/best_model.pt',  f'{SAVE_DIR}/best_model.pt')
shutil.copy('models/vocab.json',     f'{SAVE_DIR}/vocab.json')
shutil.copy('models/history.json',   f'{SAVE_DIR}/history.json')
shutil.copy('training_curves.png',   f'{SAVE_DIR}/training_curves.png')

print(f'✅ Model saved to Google Drive at: {SAVE_DIR}')
print('   Files saved:')
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(f'{SAVE_DIR}/{f}') / 1024
    print(f'   📄 {f} ({size:.1f} KB)')

---

## 📊 Model Performance Benchmarks

| Metric | Expected Range |
|--------|---------------|
| Val Accuracy | 55–63% |
| Val F1 Score | 0.52–0.62 |
| Training Time (T4 GPU) | ~5–10 min |
| Training Time (CPU) | ~20–40 min |
| Model Parameters | ~1.5M |

> 📌 **Note:** Stock direction prediction is inherently noisy. Anything above 55% sustained validation accuracy is considered strong for this task.

---

## ⚠️ Disclaimer
This project is for **educational purposes only**. It is **not financial advice**. Do not make real investment decisions based on model predictions.